In [1]:
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, rdMolAlign
from rdkit.Chem import ChemicalFeatures
from rdkit.RDConfig import RDDataDir
import os
import pandas as pd
#reading all ligands for Micobacterium tuberculosis
ligands = pd.read_excel('Mtb.xlsx', sheet_name=1)
#Name SMILES IC50 of all aurachin D homologues
ligands_Qloop = ligands[ligands['Binding site'] == 'Q-Loop']
ligands_Qloop_needed_columns = ligands_Qloop[['Name', 'SMILES', 'IC50 μM']]
#taking 1/3 of the data as training set for a model
training_set = ligands_Qloop_needed_columns[ligands_Qloop_needed_columns['IC50 μM'] < 0.3]

print(training_set)


            Name                                             SMILES  IC50 μM
2     Aurachin D  CC1=C(C(=O)C2=CC=CC=C2N1)C/C=C(\C)/CC/C=C(\C)/...    0.150
3   CK-3-22 (1T)  Cc4c(c2ccc(Oc1ccc(OC(F)(F)F)cc1)nc2)[nH]c3cccc...    0.140
8        MTD-403     Cc4c(c2ccc(N1CCCCC1)cc2)[nH]c3cc(F)cc(F)c3c4=O    0.270
9        CK-2-88          Cc4c(c2ccc(Cc1ccccc1)cc2)[nH]c3ccccc3c4=O    0.020
11       CK-2-63  Cc4c(c2ccc(Oc1ccc(OC(F)(F)F)cc1)cc2)[nH]c3cccc...    0.003
12        PG-203  Cc2[nH]c1ccccc1c(=O)c2c4ccc(Oc3ccc(OC(F)(F)F)c...    0.070
15          LT-9        O=c3cc(c2ccc(Cc1ccc(F)cc1)cc2)[nH]c4ccccc34    0.100
16        GN-171  CCOC(=O)c4c(c2ccc(Cc1ccc(OC(F)(F)F)cc1)cc2)[nH...    0.250
18       SL-2-25  Cc4c(c2ccc(c1ccc(OC(F)(F)F)cc1)nc2)[nH]c3ccccc...    0.290
19     WDH-1U-10  CCOC(=O)c4c(c2ccc(c1ccc(Cl)cc1)cc2)[nH]c3ccccc...    0.012
22      WDH-2G-6  CC(C)c4c(c2cnn(Cc1ccc(OC(F)(F)F)cc1)c2)[nH]c3c...    0.082


In [13]:
"""
Mtb Q-Loop Pharmacophore Pipeline v4
Includes: Multi-pass Sanitization, Radius Capping, and MCS Alignment.
"""

import os
import glob
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign, rdFMCS, ChemicalFeatures
from sklearn.cluster import DBSCAN
from collections import defaultdict
import warnings

# Suppress RDKit logs to keep the console clean from the sanitization noise
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

CONFORMERS_DIR   = "Conformers"
TEMPLATE_NAME    = "CK_2_63.sdf"
EXCEL_PATH       = "Mtb.xlsx"

# SCIENTIFIC TUNING
TRAINING_FRACTION = 0.5
MIN_POINT_COVERAGE = 0.4    # Require features to appear in 40% of training set
CLUSTERING_EPS = 1.5        # Tightened from 2.0 to prevent feature merging
MIN_RADIUS = 1.2
MAX_RADIUS = 2.5            # SCIENTIFIC CAP: Prevents non-selective mega-spheres
DISTANCE_CUTOFF = 1.5       # Sharper decay for better selectivity

FEATURE_FACTORY_FDEF = """
DefineFeature Donor [$([N;!H0;v3]),$([N;!H0;v4;+1]),$([n;H1;+0]),$([O,S;H1;+0])]
  Family Donor
  Weights 1.0
EndFeature
DefineFeature Acceptor [$([N;H0;+0;v3]),$([O;H0;+0;v2]),$([S;H0;+0;v2])]
  Family Acceptor
  Weights 1.0
EndFeature
DefineFeature Aromatic [$([a])]
  Family Aromatic
  Weights 1.0
EndFeature
DefineFeature Hydrophobe [$([C;v4;!$([CH2]~[O,N])]),$([c;!$([c]~[O,N])]),$([F,Cl,Br,I])]
  Family Hydrophobe
  Weights 1.0
EndFeature
"""

# ─────────────────────────────────────────────────────────────────────────────
# ROBUST LOADING & SANITIZATION
# ─────────────────────────────────────────────────────────────────────────────

def robust_sanitize(mol):
    """Attempts multi-stage sanitization to handle Kekulization/Valence errors."""
    if mol is None: return None
    try:
        # Stage 1: Standard Sanitization
        Chem.SanitizeMol(mol)
        return mol
    except:
        try:
            # Stage 2: Sanitize everything except Kekulization (Common for SDFs)
            nm = Chem.Mol(mol)
            Chem.SanitizeMol(nm, Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_KEKULIZE)
            return nm
        except:
            return None

def calculate_pIC50(ic50_uM):
    """Converts µM to pIC50: pIC50 = -log10(IC50 * 10^-6)"""
    if ic50_uM <= 0: return 0
    return -np.log10(ic50_uM * 1e-6)

def load_mol_from_sdf(path, name=None):
    # Load without initial sanitization to catch it in our robust function
    suppl = Chem.SDMolSupplier(path, removeHs=False, sanitize=False)
    mols = []
    for m in suppl:
        sanitized = robust_sanitize(m)
        if sanitized:
            # Ensure 3D conformers exist
            if sanitized.GetNumConformers() == 0:
                sanitized = Chem.AddHs(sanitized)
                AllChem.EmbedMolecule(sanitized, randomSeed=42)
            mols.append(sanitized)
    
    if not mols: return None
    
    # Merge multiple conformers from SDF into one object
    base = Chem.RWMol(mols[0])
    for extra_mol in mols[1:]:
        for conf in extra_mol.GetConformers():
            base.AddConformer(conf, assignId=True)
    
    final_mol = base.GetMol()
    if name: final_mol.SetProp("_Name", name)
    return final_mol

def find_matching_sdf(molecule_name, conformers_dir):
    name = str(molecule_name)
    candidates = [name, name.replace('-', '_'), name.replace('_', '-')]
    for c in candidates:
        p = os.path.join(conformers_dir, f"{c}.sdf")
        if os.path.exists(p): return p
    return None

# ─────────────────────────────────────────────────────────────────────────────
# CORE PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def align_to_template(probe_mol, template_mol):
    mcs = rdFMCS.FindMCS([template_mol, probe_mol], timeout=2)
    if mcs.numAtoms < 3: return probe_mol, 999.0
    
    patt = Chem.MolFromSmarts(mcs.smartsString)
    ref_match = template_mol.GetSubstructMatch(patt)
    probe_match = probe_mol.GetSubstructMatch(patt)
    
    atom_map = list(zip(probe_match, ref_match))
    best_rmsd, best_conf_id = 999.0, -1
    
    for conf_id in range(probe_mol.GetNumConformers()):
        try:
            rmsd = rdMolAlign.AlignMol(probe_mol, template_mol, prbCid=conf_id, refCid=0, atomMap=atom_map)
            if rmsd < best_rmsd:
                best_rmsd, best_conf_id = rmsd, conf_id
        except: continue
            
    if best_conf_id >= 0:
        new_mol = Chem.Mol(probe_mol)
        conf = probe_mol.GetConformer(best_conf_id)
        new_mol.RemoveAllConformers()
        new_mol.AddConformer(conf)
        return new_mol, best_rmsd
    return probe_mol, 999.0

def build_pharmacophore(aligned_mols, factory):
    all_feats = defaultdict(list)
    for idx, (mol, weight, name) in enumerate(aligned_mols):
        feats = factory.GetFeaturesForMol(mol)
        for f in feats:
            all_feats[f.GetFamily()].append({
                'pos': np.array(f.GetPos()), 'weight': weight, 'mol_idx': idx
            })
            
    model_points = []
    for family, pts in all_feats.items():
        if not pts: continue
        coords = np.array([p['pos'] for p in pts])
        weights = np.array([p['weight'] for p in pts])
        
        clusters = DBSCAN(eps=CLUSTERING_EPS, min_samples=2).fit(coords)
        for label in set(clusters.labels_):
            if label == -1: continue
            mask = clusters.labels_ == label
            
            # Weighted center based on pIC50
            c_coords = coords[mask]
            c_weights = weights[mask]
            center = np.average(c_coords, axis=0, weights=c_weights)
            
            # Radius calculation with scientific capping
            raw_radius = np.max(np.linalg.norm(c_coords - center, axis=1))
            radius = clip_radius = min(max(raw_radius, MIN_RADIUS), MAX_RADIUS)
            
            unique_mols = len(set(pts[i]['mol_idx'] for i in np.where(mask)[0]))
            conservation = unique_mols / len(aligned_mols)
            
            if conservation >= MIN_POINT_COVERAGE:
                model_points.append({'family': family, 'center': center, 'radius': radius, 'cons': conservation})
    
    return sorted(model_points, key=lambda x: -x['cons'])

def score_ligands(ligands, points, factory):
    results = []
    for name, mol, ic50 in ligands:
        mol_feats = factory.GetFeaturesForMol(mol)
        feat_map = defaultdict(list)
        for f in mol_feats: feat_map[f.GetFamily()].append(np.array(f.GetPos()))
        
        point_scores = []
        for pt in points:
            if pt['family'] in feat_map:
                dists = [np.linalg.norm(pos - pt['center']) for pos in feat_map[pt['family']]]
                min_d = min(dists)
                # Soft-Top Gaussian
                s = 1.0 if min_d <= pt['radius'] else np.exp(-((min_d - pt['radius'])**2) / (2 * (DISTANCE_CUTOFF**2)))
                point_scores.append(s)
            else: point_scores.append(0.0)
            
        fit = np.mean(point_scores) if point_scores else 0
        results.append({'name': name, 'ic50': ic50, 'fit': fit})
    return results

# ─────────────────────────────────────────────────────────────────────────────
# MAIN EXECUTION
# ─────────────────────────────────────────────────────────────────────────────

def run():
    print("="*50 + "\nMTB Q-LOOP PHARMACOPHORE PIPELINE\n" + "="*50)
    
    # 1. Load Activity Data
    df = pd.read_excel(EXCEL_PATH, sheet_name=1)
    df = df[df['Binding site'].str.upper() == 'Q-LOOP'].copy()
    df['pIC50'] = pd.to_numeric(df['IC50 μM'].astype(str).str.replace('μM',''), errors='coerce').apply(calculate_pIC50)
    df = df.dropna(subset=['pIC50']).sort_values('pIC50', ascending=False)
    
    train_df = df.head(int(len(df)*TRAINING_FRACTION))
    val_df = df.tail(len(df)-len(train_df))
    
    # 2. Setup Factory and Template
    factory = ChemicalFeatures.BuildFeatureFactoryFromString(FEATURE_FACTORY_FDEF)
    template = load_mol_from_sdf(os.path.join(CONFORMERS_DIR, TEMPLATE_NAME), "TEMPLATE")
    
    # 3. Align Training Set
    aligned_mols = []
    print(f"Aligning {len(train_df)} training molecules...")
    for _, row in train_df.iterrows():
        path = find_matching_sdf(row['Name'], CONFORMERS_DIR)
        if path:
            m = load_mol_from_sdf(path, row['Name'])
            if m:
                ready, rmsd = align_to_template(m, template)
                if rmsd < 5.0:
                    aligned_mols.append((ready, row['pIC50'], row['Name']))
    
    # 4. Generate Model
    ph_model = build_pharmacophore(aligned_mols, factory)
    print(f"\nModel generated with {len(ph_model)} features:")
    for i, p in enumerate(ph_model, 1):
        print(f"  {i}. {p['family']:12} | Cons: {p['cons']:.2f} | Radius: {p['radius']:.2f} Å")
    
    # 5. Validation
    val_data = []
    for _, row in val_df.iterrows():
        path = find_matching_sdf(row['Name'], CONFORMERS_DIR)
        if path:
            m = load_mol_from_sdf(path, row['Name'])
            if m: val_data.append((row['Name'], m, 10**(-row['pIC50']+6))) # Back to uM
            
    results = score_ligands(val_data, ph_model, factory)
    
    print(f"\n{'Name':<18} {'IC50 (μM)':<12} {'Fit Score':<10}")
    print("-" * 45)
    for r in sorted(results, key=lambda x: x['ic50']):
        print(f"{r['name']:<18} {r['ic50']:<12.4f} {r['fit']:<10.2f}")

    if results:
        active_fit = np.mean([r['fit'] for r in results if r['ic50'] <= 1.0])
        print(f"\nMean Active Fit Score: {active_fit:.2f}")

if __name__ == "__main__":
    run()

MTB Q-LOOP PHARMACOPHORE PIPELINE
Aligning 11 training molecules...

Model generated with 4 features:
  1. Hydrophobe   | Cons: 1.00 | Radius: 2.50 Å
  2. Donor        | Cons: 0.67 | Radius: 1.20 Å
  3. Acceptor     | Cons: 0.67 | Radius: 1.20 Å
  4. Acceptor     | Cons: 0.44 | Radius: 1.20 Å

Name               IC50 (μM)    Fit Score 
---------------------------------------------
RKA-73             0.3100       0.56      
WDH-2R-4           0.3800       0.67      
RKA-307            0.4400       0.70      
RKA-70             0.7500       0.29      
RKA-310            1.4000       0.68      
CK-3-23            3.6000       0.29      
PG-128             4.4700       0.62      
WDH-2A-9           6.5000       0.67      
CK-3-14            10.6000      0.64      

Mean Active Fit Score: 0.55
